# Regression Trees, Random Forest and Gradient Boosting – Cycling at the Lake of Zurich

**Dataset:** Bike counts in Zurich 2024–2025: daily number of bikes at 16 automatic counting stations of the city of
Zurich, with weather and calendar information (Open Data Stadt Zürich). File: `../00_Data/zurich_bike_counts_2024_2025.csv`.
We model the counting station **Mythenquai** – a cycle route along the lake between the city centre and Wollishofen,
used both by commuters and for leisure.

**Business question:** How many cyclists will pass the Mythenquai on a given day, depending on weather, weekday and
holidays? Weather and weekday interact (a sunny Sunday attracts many leisure cyclists, a sunny Tuesday mainly commuters)
– a typical use case for tree-based models.

**After this exercise you can ...**
- find the best split of a regression tree by hand (SSE criterion),
- fit, visualise and prune a regression tree,
- explain and apply bagging, random forests (incl. out-of-bag error, permutation importance) and gradient boosting,
- compare models fairly with cross-validation.

## Concept

The concepts behind this exercise (formulas, worked examples and additional explanations) are
explained in the notebook [regression_trees_ensembles_concept.ipynb](regression_trees_ensembles_concept.ipynb). Read it before you start.

## Libraries and settings

In [ ]:
# Libraries
import os
import platform
import warnings
from datetime import datetime
from platform import python_version

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn import tree
from sklearn.tree import DecisionTreeRegressor
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.model_selection import (
    train_test_split,
    cross_val_score,
    GridSearchCV,
    KFold,
)
from sklearn.metrics import root_mean_squared_error, r2_score
from sklearn.inspection import permutation_importance

# Ignore warnings
warnings.filterwarnings('ignore')

# Show current working directory
print(os.getcwd())

## Exercise 1: Calculations by hand

**a) Best split of a regression tree.** On six days the following numbers of bikes (in hundreds) were counted at the
Mythenquai:

| daily mean temperature $x$ [°C] | −2 | 3 | 8 | 14 | 20 | 26 |
|---|---|---|---|---|---|---|
| bikes $y$ [hundreds] | 15 | 22 | 30 | 38 | 42 | 44 |

Compute the SSE of the root node. Then compute, for each of the five possible thresholds (midpoints between neighbouring
temperatures), the means and the SSE of the two child nodes. Which split does the tree choose? What are the leaf
predictions?

**b) Variance of an average.** 100 trees have variance $\sigma^2 = 4$ each. Compute the variance of their average for
$\rho = 0.6$ (bagged trees, similar to each other) and for $\rho = 0.2$ (random forest). What happens for
$B \to \infty$?

**c) One boosting step.** The current prediction for a day is 30 (hundred bikes), the observed number is 40. The next
tree predicts the residual exactly. What is the new prediction with learning rate $\nu = 0.1$? How many such steps are
needed until the remaining residual is below 5?

**d)** Verify your results with Python.

In [ ]:
# a) Best split
x = np.array([-2, 3, 8, 14, 20, 26])
y = np.array([15, 22, 30, 38, 42, 44])


# Function: computes the sum of squared deviations of v from its mean
def sse(v):
    """Sum of squared errors around the mean."""
    return np.sum((v - v.mean()) ** 2)


# SSE of the root and of all possible splits
print(f'Root: mean = {y.mean():.2f}, SSE = {sse(y):.2f}')
rows = []
for s in (x[:-1] + x[1:]) / 2:
    left, right = y[x <= s], y[x > s]
    rows.append(
        {
            'split': f'x <= {s}',
            'mean left': left.mean(),
            'mean right': right.mean(),
            'SSE left': sse(left),
            'SSE right': sse(right),
            'SSE total': sse(left) + sse(right),
        }
    )
print(pd.DataFrame(rows).round(2).to_string(index=False))

# Check with scikit-learn: a tree with one split (a "stump")
stump = DecisionTreeRegressor(max_depth=1).fit(x.reshape(-1, 1), y)
print(tree.export_text(stump, feature_names=['temperature']))

# b) Variance of an average
sigma2, B = 4, 100
for rho in [0.6, 0.2]:
    print(
        f'rho = {rho}: Var = {rho * sigma2 + (1 - rho) / B * sigma2:.3f}  '
        f'(limit for B -> inf: {rho * sigma2})'
    )

# c) Boosting steps
y_obs, pred, nu, steps = 40, 30, 0.1, 0
while y_obs - pred >= 5:
    pred = pred + nu * (y_obs - pred)
    steps += 1
    if steps == 1:
        print('Prediction after 1 step:', pred)
print(f'Steps until residual < 5: {steps} (prediction {pred:.2f})')

**Solution:**

a) Root: $\bar y = 191/6 \approx 31.83$, SSE $\approx 672.83$.

| split | left node | mean | right node | mean | SSE left + SSE right |
|---|---|---|---|---|---|
| $x \le 0.5$ | {15} | 15 | {22, 30, 38, 42, 44} | 35.2 | 0 + 332.8 = 332.8 |
| $x \le 5.5$ | {15, 22} | 18.5 | {30, 38, 42, 44} | 38.5 | 24.5 + 115 = 139.5 |
| $x \le 11$ | {15, 22, 30} | 22.33 | {38, 42, 44} | 41.33 | 112.67 + 18.67 = **131.33** |
| $x \le 17$ | {15, 22, 30, 38} | 26.25 | {42, 44} | 43 | 296.75 + 2 = 298.75 |
| $x \le 23$ | {15, …, 42} | 29.4 | {44} | 44 | 495.2 + 0 = 495.2 |

The tree chooses **temperature ≤ 11 °C** (SSE drops from 672.8 to 131.3, −80 %). Leaf predictions: 2'233 bikes on cold
days and 4'133 bikes on warm days.

b) $\rho = 0.6$: $0.6\cdot 4 + 0.4/100\cdot 4 = 2.416$; $\rho = 0.2$: $0.2\cdot 4 + 0.8/100\cdot 4 = 0.832$.
For $B \to \infty$ only the shared part $\rho\sigma^2$ remains (2.4 resp. 0.8). More trees cannot remove it – this is
why the random forest de-correlates the trees.

c) Residual 10 → new prediction $30 + 0.1\cdot 10 = 31$. Each step removes 10 % of the remaining residual:
$10\cdot 0.9^m < 5$ → $m = 7$ steps (residual $10\cdot 0.9^7 \approx 4.78$). A small learning rate needs many trees but
makes the model less prone to overfitting.

## Exercise 2: Import and explore the bike counts at the Mythenquai

a) Import the bike counts. The target is the column `mythenquai` (bikes per day at this station). Use the features
`temp_mean`, `temp_max`, `rain_duration_min`, `solar_radiation`, `month`, `holiday`, `school_holiday` and the
`weekday` (as dummy variables: `pd.get_dummies(..., columns=['weekday'], dtype=int)`).

b) Plot `mythenquai` against `temp_max`, separately for working days and weekends. What do you see?

c) Split the data into training (80 %) and test set (20 %) with `random_state=42`.

In [ ]:
# a) Import the data
bikes = pd.read_csv('../00_Data/zurich_bike_counts_2024_2025.csv', sep=',')

# Features (weekday as dummy variables) and target
features = [
    'temp_mean',
    'temp_max',
    'rain_duration_min',
    'solar_radiation',
    'month',
    'holiday',
    'school_holiday',
    'weekday',
]
X = pd.get_dummies(bikes[features], columns=['weekday'], dtype=int)
y = bikes['mythenquai']
print(X.shape)
print(y.describe().round(0))

# b) Scatter plot by working day / weekend
plt.figure(figsize=(8, 4.5))
for w, label, color in [
    (0, 'working day', 'tab:blue'),
    (1, 'weekend', 'tab:red'),
]:
    m = bikes['weekend'] == w
    plt.scatter(
        bikes.loc[m, 'temp_max'],
        y[m],
        s=12,
        alpha=0.6,
        color=color,
        label=label,
    )
plt.xlabel('Maximum temperature [°C]')
plt.ylabel('Bikes per day at the Mythenquai')
plt.legend()
plt.grid(alpha=0.3)
plt.show()

# c) Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)
print('Training set:', X_train.shape, ' Test set:', X_test.shape)

**Solution:**

a) 674 days with 14 features (7 weather/calendar columns + 7 weekday dummies). On average about 1'830 bikes per day
pass the Mythenquai (between about 150 on cold, rainy holidays and more than 6'000 on hot summer days).

b) The number of bikes increases strongly with temperature, and the increase is **not linear** (steeper above about
20 °C – leisure traffic to the lake and the lidos). The difference between working days and weekends **depends on the
weather**: on cold days, the weekend count is only about half of the working-day count (≈ 580 vs. ≈ 1'000), on hot days
(≥ 25 °C) both are about equal (≈ 3'500). Such an **interaction** cannot be captured by a linear model with additive
effects, but it is natural for a tree.

## Exercise 3: Baseline – linear regression

Fit a multiple linear regression with all features on the training set and compute RMSE and $R^2$ on the test set. This
is the baseline that the tree-based models have to beat.

In [ ]:
# Exercise 3: linear regression with all features; test RMSE and R²
lin_reg = LinearRegression().fit(X_train, y_train)
y_pred_lin = lin_reg.predict(X_test)
print(
    'Linear regression - test RMSE:',
    round(root_mean_squared_error(y_test, y_pred_lin)),
    ' test R2:',
    round(r2_score(y_test, y_pred_lin), 3),
)

## Exercise 4: A small regression tree

a) Fit a `DecisionTreeRegressor(max_depth=3, random_state=42)` on the training set.

b) Visualise the tree with `tree.plot_tree()` and interpret the first two levels. Which features are used?

c) Compute RMSE and $R^2$ on the test set. How does the small tree compare to the linear regression?

In [ ]:
# a) Fit the tree
reg_tree = DecisionTreeRegressor(max_depth=3, random_state=42).fit(
    X_train, y_train
)

# b) Plot the tree
plt.figure(figsize=(20, 7))
tree.plot_tree(
    reg_tree,
    feature_names=list(X_train.columns),
    filled=True,
    rounded=True,
    fontsize=9,
    precision=1,
)
plt.show()

# c) Test RMSE and R2
y_pred_tree = reg_tree.predict(X_test)
print(
    'Tree (depth 3) - test RMSE:',
    round(root_mean_squared_error(y_test, y_pred_tree)),
    ' test R2:',
    round(r2_score(y_test, y_pred_tree), 3),
)

**Solution:**

The tree splits first on **solar radiation** (≤ 180 W/m², i.e. cloudy vs. sunny days) and then on the **maximum
temperature** (around 13 °C on cloudy days, around 25 °C on sunny days). Further down it uses the temperature again and
the **school holidays** (cold days during the school holidays: only ≈ 590 bikes – many commuters are away). The leaves
range from ≈ 590 to ≈ 4'300 bikes: "sunny and above 29 °C → ≈ 4'300 bikes".

With only 8 leaves the small tree has a test RMSE of ≈ 580 ($R^2 \approx 0.72$) and is **worse** than the linear
regression (≈ 490, $R^2 \approx 0.80$): it is easy to interpret but has a high bias – it predicts only 8 different values.

## Exercise 5: Tree complexity and pruning

a) For `max_depth` = 1, 2, …, 15 compute the RMSE on the training set and the 5-fold CV RMSE (on the training set).
Plot both curves. Where does the tree start to overfit?

b) Alternatively, grow a full tree and prune it: compute the candidate values with
`DecisionTreeRegressor(random_state=42).cost_complexity_pruning_path(X_train, y_train).ccp_alphas` and choose `ccp_alpha`
with `GridSearchCV` (5-fold CV, `scoring='neg_root_mean_squared_error'`). How many leaves does the best pruned tree have?
What is its test RMSE?

In [ ]:
# Exercise 5: tree complexity and pruning
cv = KFold(n_splits=5, shuffle=True, random_state=42)

# a) Training vs. CV error for different depths
depths = range(1, 16)
rmse_train, rmse_cv = [], []
for d in depths:
    m = DecisionTreeRegressor(max_depth=d, random_state=42)
    rmse_cv.append(
        -cross_val_score(
            m, X_train, y_train, cv=cv, scoring='neg_root_mean_squared_error'
        ).mean()
    )
    rmse_train.append(
        root_mean_squared_error(
            y_train, m.fit(X_train, y_train).predict(X_train)
        )
    )

# Figure showing the training and CV error by tree depth
plt.figure(figsize=(7, 4))
plt.plot(depths, rmse_train, marker='o', label='training error')
plt.plot(depths, rmse_cv, marker='o', label='5-fold CV error')
plt.xlabel('max_depth')
plt.ylabel('RMSE [bikes per day]')
plt.legend()
plt.grid(alpha=0.3)
plt.show()
print(
    'Best depth by CV:',
    depths[int(np.argmin(rmse_cv))],
    ' CV RMSE:',
    round(min(rmse_cv)),
)

# b) Cost-complexity pruning
alphas = (
    DecisionTreeRegressor(random_state=42)
    .cost_complexity_pruning_path(X_train, y_train)
    .ccp_alphas
)
grid_tree = GridSearchCV(
    DecisionTreeRegressor(random_state=42),
    {'ccp_alpha': alphas},
    cv=cv,
    scoring='neg_root_mean_squared_error',
)
grid_tree.fit(X_train, y_train)
pruned_tree = grid_tree.best_estimator_
print('Number of candidate alphas:', len(alphas))
print(
    'Best ccp_alpha:',
    round(grid_tree.best_params_['ccp_alpha'], 1),
    ' CV RMSE:',
    round(-grid_tree.best_score_),
)
print(
    'Leaves: full tree =',
    DecisionTreeRegressor(random_state=42).fit(X_train, y_train).get_n_leaves(),
    ' pruned tree =',
    pruned_tree.get_n_leaves(),
)
print(
    'Pruned tree - test RMSE:',
    round(root_mean_squared_error(y_test, pruned_tree.predict(X_test))),
)

**Solution:**

a) The training error decreases steadily with the depth and approaches 0 for deep trees (each leaf contains only very
few days). The CV error is lowest at **depth 3** (≈ 570) and then increases again – the additional splits only learn the
noise of single days (high variance). With only 539 training days, a single tree overfits quickly.

b) Pruning with cross-validation leads to the same conclusion: of about 530 leaves of the full tree, only **8 leaves**
remain (CV RMSE ≈ 580, test RMSE ≈ 580). A single tree – however well tuned – remains clearly worse than the linear
regression here.

## Exercise 6: Random forest

a) Fit random forests with `n_estimators` = 1, 5, 10, 25, 50, 100, 200, 400 (`oob_score=True`, `random_state=42`,
`n_jobs=-1`). The OOB predictions are stored in `oob_prediction_`. Plot the OOB RMSE against the number of trees. When do
the gains plateau?

b) Compare `max_features` = 1.0 (all features, i.e. bagging), `'sqrt'` and 0.5 with 5-fold CV (300 trees).

c) For the best forest, compute the test RMSE. Compare the impurity-based feature importance (`feature_importances_`)
with the **permutation importance** on the test set.

In [ ]:
# a) Number of trees vs. OOB error
n_trees = [1, 5, 10, 25, 50, 100, 200, 400]
oob_rmse = []
for n in n_trees:
    rf = RandomForestRegressor(
        n_estimators=n, oob_score=True, random_state=42, n_jobs=-1
    )
    rf.fit(X_train, y_train)
    pred_oob = rf.oob_prediction_
    ok = ~np.isnan(
        pred_oob
    )  # with very few trees, some days are never out-of-bag
    oob_rmse.append(root_mean_squared_error(y_train[ok], pred_oob[ok]))

# Figure showing the OOB RMSE by number of trees
plt.figure(figsize=(7, 4))
plt.plot(n_trees, oob_rmse, marker='o')
plt.xscale('log')
plt.xlabel('n_estimators (log scale)')
plt.ylabel('OOB RMSE [bikes per day]')
plt.title('Random forest: OOB error vs. number of trees')
plt.grid(alpha=0.3)
plt.show()
print(
    pd.DataFrame(
        {'n_estimators': n_trees, 'OOB RMSE': np.round(oob_rmse)}
    ).to_string(index=False)
)

# b) max_features
rf_cv = {}
for mf in [1.0, 'sqrt', 0.5]:
    rf = RandomForestRegressor(
        n_estimators=300, max_features=mf, random_state=42, n_jobs=-1
    )
    rf_cv[mf] = -cross_val_score(
        rf, X_train, y_train, cv=cv, scoring='neg_root_mean_squared_error'
    ).mean()
    print(f'max_features = {mf}: CV RMSE = {rf_cv[mf]:.0f}')

In [ ]:
# c) Test RMSE and feature importance (use the best max_features from b)
best_mf = min(rf_cv, key=rf_cv.get)
rf_best = RandomForestRegressor(
    n_estimators=300, max_features=best_mf, random_state=42, n_jobs=-1
)
rf_best.fit(X_train, y_train)
print(
    f'Random forest (max_features = {best_mf}) - test RMSE:',
    round(root_mean_squared_error(y_test, rf_best.predict(X_test))),
    ' test R2:',
    round(r2_score(y_test, rf_best.predict(X_test)), 3),
)

# Permutation importance on the test set vs. impurity-based importance
perm = permutation_importance(
    rf_best, X_test, y_test, n_repeats=10, random_state=42, n_jobs=-1
)
imp = pd.DataFrame(
    {
        'impurity-based (MDI)': rf_best.feature_importances_,
        'permutation (test set)': perm.importances_mean,
    },
    index=X_train.columns,
).sort_values('permutation (test set)')
print(imp.round(3))

# Figure comparing both importance measures
fig, axes = plt.subplots(1, 2, figsize=(13, 5), sharey=True)
imp['impurity-based (MDI)'].plot(
    kind='barh', ax=axes[0], color='grey', title='Impurity-based importance'
)
imp['permutation (test set)'].plot(
    kind='barh',
    ax=axes[1],
    color='darkred',
    title='Permutation importance (decrease in R²)',
)
plt.show()

**Solution:**

a) A single tree has a very high OOB error (≈ 1'800); the error drops quickly with the first 25 trees and **plateaus**
at about 450–460 after 50–100 trees. More trees never hurt the accuracy, they only cost computing time.

b) Considering only a part of the features at each split (`max_features=0.5` or `'sqrt'`, CV RMSE ≈ 440) is slightly
better than plain bagging (`1.0`, ≈ 455), because the trees become less correlated. (Note: scikit-learn uses all
features by default for regression.)

c) The random forest reaches a test RMSE of ≈ 435 ($R^2 \approx 0.84$) – clearly better than the single tree and also
better than the linear regression. The most important features are **solar radiation** and **maximum temperature**,
followed by rain, school holidays and the mean temperature. The two importance measures differ: impurity-based importance
gives `temp_mean` a high weight, while permutation importance shows that it adds little once `temp_max` is known (the two
temperatures are strongly correlated and share their importance). Saturday and Sunday matter more in the permutation
importance than in MDI. Permutation importance is computed on the test set and is therefore the more reliable measure.

## Exercise 7: Gradient boosting

a) Fit `GradientBoostingRegressor(n_estimators=1000, max_depth=3, random_state=42)` with learning rates
$\nu$ = 0.01, 0.1 and 1.0. Use `staged_predict(X_test)` to compute the test RMSE after each boosting iteration and plot
the three curves.

*Note: we look at the test curve here only for illustration. In practice, `learning_rate` and `n_estimators` are tuned
with cross-validation (see Exercise 8).*

b) Describe the influence of the learning rate. What happens with $\nu = 1.0$?

In [ ]:
# a) Test error per iteration for different learning rates
plt.figure(figsize=(8, 4))
for lr in [0.01, 0.1, 1.0]:
    gb = GradientBoostingRegressor(
        n_estimators=1000, max_depth=3, learning_rate=lr, random_state=42
    )
    gb.fit(X_train, y_train)
    rmse_staged = [
        root_mean_squared_error(y_test, p) for p in gb.staged_predict(X_test)
    ]
    plt.plot(range(1, 1001), rmse_staged, label=f'learning rate {lr}')
    print(
        f'learning rate {lr}: min test RMSE = {min(rmse_staged):.0f} '
        f'after {int(np.argmin(rmse_staged)) + 1} trees,'
        f' after 1000 trees = {rmse_staged[-1]:.0f}'
    )
plt.ylim(0, 1000)
plt.xlabel('Number of trees M')
plt.ylabel('Test RMSE [bikes per day]')
plt.title('Gradient boosting: test error vs. number of trees')
plt.legend()
plt.grid(alpha=0.3)
plt.show()

**Solution:**

- $\nu = 0.01$: very small steps – the error decreases slowly and is still decreasing after 1'000 trees (≈ 415).
- $\nu = 0.1$: the error decreases quickly and reaches the lowest value (≈ 390); this is the default and usually a good
  compromise.
- $\nu = 1.0$: each tree corrects the full residual. The error drops very fast at the beginning, reaches its minimum
  (≈ 475) after only about 15 trees and then **rises again** (≈ 545 after 1'000 trees) – the model fits the noise of
  the training data (overfitting).

Learning rate and number of trees must be chosen **together**: a smaller $\nu$ needs a larger $M$.

## Exercise 8: Fair model comparison

Compare the following models with **5-fold CV RMSE on the training set** and summarise the results in a table:
linear regression, pruned tree (best `ccp_alpha` from Exercise 5), random forest (best setting from Exercise 6) and
gradient boosting (`learning_rate=0.05`, `n_estimators=300`, `max_depth=3`). Then evaluate the best model **once** on the
test set. Which model would you recommend to the city, and what are the drawbacks?

In [ ]:
# Exercise 8: compare all models with the same cross-validation
models = {
    'Linear regression': LinearRegression(),
    'Pruned tree': DecisionTreeRegressor(
        ccp_alpha=grid_tree.best_params_['ccp_alpha'], random_state=42
    ),
    'Random forest': RandomForestRegressor(
        n_estimators=300, max_features=best_mf, random_state=42, n_jobs=-1
    ),
    'Gradient boosting': GradientBoostingRegressor(
        learning_rate=0.05, n_estimators=300, max_depth=3, random_state=42
    ),
}

# 5-fold CV RMSE of all models
rows = []
for name, m in models.items():
    scores = -cross_val_score(
        m, X_train, y_train, cv=cv, scoring='neg_root_mean_squared_error'
    )
    rows.append(
        {
            'model': name,
            'CV RMSE (mean)': scores.mean(),
            'CV RMSE (std)': scores.std(),
        }
    )
comparison = pd.DataFrame(rows).sort_values('CV RMSE (mean)')
print(comparison.round(0).to_string(index=False))

# Fit the best model on the training set, evaluate on the test set
best_name = comparison.iloc[0]['model']
best_model = models[best_name].fit(X_train, y_train)
print(f'\nBest model by CV: {best_name}')
print(
    'Test RMSE:',
    round(root_mean_squared_error(y_test, best_model.predict(X_test))),
    ' test R2:',
    round(r2_score(y_test, best_model.predict(X_test)), 3),
)

**Solution:**

| model | CV RMSE |
|---|---|
| Gradient boosting | ≈ 417 |
| Random forest | ≈ 441 |
| Linear regression | ≈ 528 |
| Pruned tree | ≈ 582 |

The ensembles are clearly best; gradient boosting wins narrowly against the random forest and reaches a test RMSE of
≈ 410 ($R^2 \approx 0.86$). The single tree is the worst model – but it is the building block that makes the ensembles
so strong. Note the large standard deviation of the CV RMSE (≈ 80): with only ~540 training days, the difference
between the two ensembles is not very reliable.

Recommendation: gradient boosting or random forest for the prediction. Drawbacks: the models are black boxes (only
feature importance, no simple formula or rules), they cannot extrapolate beyond the range of the training data (e.g. a
record heat wave or a new cycle lane), and gradient boosting needs careful tuning of learning rate, number and depth of
trees. If an easily communicable model is needed, the linear regression is a reasonable compromise.

Note: ensembles do not always win. For data with largely additive, roughly linear relationships (e.g. rents explained
by living area and location) a linear regression is often almost as good – always compare with cross-validation.

## Exercise 9: More covariates

So far the models used the weather (temperature, rain, sun) and the calendar (month, weekday, holidays). The data set
contains more information: the air pressure (`pressure`), the exact position in the year (`day_of_year`, computed from
`date`) and the `year` (2024 or 2025).

a) Add these three covariates and split the data again (80/20, `random_state=42` – the same days as before).

b) Compute the 5-fold CV RMSE of the linear regression, the random forest (best setting of Exercise 6) and gradient
boosting (as in Exercise 8) with the **extended** feature set and compare with Exercise 8.

c) Evaluate the best model on the test set and compute its permutation importance. Which new covariate matters most, and
why?

In [ ]:
# a) Extended feature set
bikes['day_of_year'] = pd.to_datetime(bikes['date']).dt.dayofyear
features_more = features + ['pressure', 'day_of_year', 'year']
X_more = pd.get_dummies(bikes[features_more], columns=['weekday'], dtype=int)
X2_train, X2_test, y2_train, y2_test = train_test_split(
    X_more, y, test_size=0.20, random_state=42
)
print(
    X_more.shape,
    ' same test days as before:',
    (X2_test.index == X_test.index).all(),
)

# b) CV RMSE with the extended feature set
rows = []
for name, m in models.items():
    if name == 'Pruned tree':
        continue
    scores = -cross_val_score(
        m, X2_train, y2_train, cv=cv, scoring='neg_root_mean_squared_error'
    )
    before = comparison.loc[comparison['model'] == name, 'CV RMSE (mean)'].iloc[
        0
    ]
    rows.append(
        {
            'model': name,
            'CV RMSE (Exercise 8)': round(before),
            'CV RMSE (more covariates)': round(scores.mean()),
        }
    )
print(pd.DataFrame(rows).to_string(index=False))

# c) Test RMSE and permutation importance of the best model
gb_more = GradientBoostingRegressor(
    learning_rate=0.05, n_estimators=300, max_depth=3, random_state=42
)
gb_more.fit(X2_train, y2_train)
print(
    'Gradient boosting (more covariates) - test RMSE:',
    round(root_mean_squared_error(y2_test, gb_more.predict(X2_test))),
    ' test R2:',
    round(r2_score(y2_test, gb_more.predict(X2_test)), 3),
)
perm = permutation_importance(
    gb_more, X2_test, y2_test, n_repeats=10, random_state=42, n_jobs=-1
)
imp = pd.Series(perm.importances_mean, index=X_more.columns).sort_values()
imp.tail(10).plot(
    kind='barh',
    color='darkred',
    figsize=(7, 4),
    title='Permutation importance (test set)',
)
plt.show()
print(imp.sort_values(ascending=False).head(8).round(3))

**Solution:**

b)
| model | CV RMSE (Exercise 8) | CV RMSE (more covariates) |
|---|---|---|
| linear regression | ≈ 528 | ≈ 526 |
| random forest | ≈ 441 | ≈ 404 |
| gradient boosting | ≈ 417 | ≈ 363 |

The tree ensembles profit clearly (−8 % and −13 %), the linear regression hardly at all.

c) Gradient boosting with the extended feature set reaches a test RMSE of ≈ 380 bikes per day ($R^2 \approx 0.88$; before
≈ 410 and 0.86). The most important new covariate is **`day_of_year`** (third most important feature after solar
radiation and maximum temperature). It describes the season much more finely than `month` – e.g. the start of the
summer holidays, the long evenings in June or the first cold days in autumn – and the trees can model its **non-linear**
effect. For the linear regression a linear trend over the year is useless (the effect rises in spring and falls in
autumn), which is why it does not profit. `pressure` and `year` add little: the weather is already described by
temperature, sun and rain, and the difference between 2024 and 2025 is small compared to the weather effects.

Lesson: **better covariates often help more than more tuning** – but only if the model can use them (non-linear effects
need flexible models or suitable transformations).

### Jupyter notebook --footer info-- (please always provide this at the end of each notebook)

In [ ]:
# Show operating system, date/time and Python version
print('-----------------------------------')
print(os.name.upper())
print(platform.system(), '|', platform.release())
print('Datetime:', datetime.now().strftime("%Y-%m-%d %H:%M:%S"))
print('Python Version:', python_version())
print('-----------------------------------')